## Helper Functions

In [ ]:
def ask_question(session, question):
    """Ask a question and display the response.

    Parameters
    ----------
    session : RAGSession
        Active KíLín session.
    question : str
        Question to send to the model.

    Returns
    -------
    str
        Model response.

    """

    print(f"\n{'=' * 60}")
    print(f"USER: {question}")
    print("=" * 60)

    answer = session.ask(question)

    print(f"\nASSISTANT:\n{answer}")

    return answer

In [ ]:
import json
from pathlib import Path


def inspect_conversation(conversation_path, turn_number=None):
    """Inspect the recorded conversation and routing decisions.

    Parameters
    ----------
    conversation_path : str or Path
        Path to the saved conversation JSON.
    turn_number : int, optional
        One-based turn number to inspect. If None, inspect all turns.

    Returns
    -------
    dict
        Complete conversation trace.

    """
    conversation_path = Path(conversation_path)

    with conversation_path.open(encoding="utf-8") as file:
        trace = json.load(file)

    turns = trace["sessions"]["05_conversations"]["turns"]
    selected = list(turns.items())

    if turn_number is not None:
        if not 1 <= turn_number <= len(selected):
            raise ValueError("Invalid turn number.")
        selected = [selected[turn_number - 1]]

    for turn_id, turn in selected:
        print(f"\n{'=' * 60}")
        print(f"TURN: {turn_id}")
        print("=" * 60)

        stages = turn.get("stages", {})
        question_stage = stages.get("01_question", {})
        operations = question_stage.get("operations", {})

        decision = operations.get(
            "05_intent_classification_decided", {}
        ).get("details", {})

        classification = decision.get("classification", {})
        resolved = decision.get("resolved", {})

        print("\nINTENT CLASSIFICATION")
        print("  Intent:", classification.get("intent"))
        print("  Standalone:", classification.get("standalone"))
        print("  Refers back:", classification.get("refers_back"))
        print("  Related turns:", classification.get("related_turn_ids"))
        print("  Shortcut:", decision.get("shortcut"))
        print("  Fallback:", decision.get("fallback"))

        print("\nROUTING POLICY")
        print("  History:", resolved.get("history_policy"))
        print("  Retrieval mode:", resolved.get("retrieval_mode"))
        print("  Retrieval strategy:", resolved.get("retrieval_strategy"))

        print("\nRECORDED OPERATIONS")
        for stage_name, stage in stages.items():
            print(f"  {stage_name}")
            for operation_name in stage.get("operations", {}):
                print(f"    {operation_name}")

    return trace

## Create Empty Literature Project

In [ ]:
from pathlib import Path

import kilin

# Locate the repository independently of the notebook working directory.
repo_root = Path.cwd().resolve()
while not (repo_root / "pyproject.toml").is_file():
    if repo_root == repo_root.parent:
        raise RuntimeError("Run this notebook from within the KíLín repository.")
    repo_root = repo_root.parent

project_dir = repo_root / "sandbox" / "trial_literature"
project_dir.parent.mkdir(parents=True, exist_ok=True)

# Do not overwrite a project created by an earlier run.
if project_dir.exists():
    raise FileExistsError(f"Project directory already exists: {project_dir}")

project = kilin.initialize_project(
    project_dir,
    kilin.PROJECT_TYPE_ID_LITERATURE,
)

print(f"Created literature project: {project_dir}")

## First Conversation without Sources

In [ ]:
from kilin.conversations import ConversationStore
from kilin.rag import RAGSession

project_dir = repo_root / "sandbox" / "trial_literature"

# Create a new, independent conversation.
store = ConversationStore(project_dir)
conversation = store.create()

# Load the project and run the actual model.
session = RAGSession(
    project_dir,
    conversation_path=store.state_path(conversation),
)

print(f"\nConversation saved to: {store.state_path(conversation)}")

In [ ]:
answer = ask_question(session, "hello")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=1,
)

In [ ]:
answer = ask_question(session, "What can you help me with?")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=2,
)

In [ ]:
session.close()

## Add Articles to Project